In [3]:
import os
import ffmpeg
import gradio as gr
from utils_transcripcion import obtener_o_transcribir_audio, extraer_audio_con_ffmpeg
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
import datetime
from transformers import BlipProcessor, BlipForConditionalGeneration
from transformers import BridgeTowerProcessor, BridgeTowerModel
from PIL import Image
import torch
import cv2
import numpy as np
import chromadb
from chromadb.utils import embedding_functions
from chromadb.config import Settings
from datetime import timedelta

transcripciones_cache = {}
historial = []

c:\Users\Arria\anaconda3\envs\libretainteligente\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Modelos
device = "cuda" if torch.cuda.is_available() else "cpu"

# BLIP para captions
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

# BridgeTower para embeddings multimodales
bridge_processor = BridgeTowerProcessor.from_pretrained("BridgeTower/bridgetower-base")
bridge_model = BridgeTowerModel.from_pretrained("BridgeTower/bridgetower-base").to(device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
c:\Users\Arria\anaconda3\envs\libretainteligente\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Arria\.cache\huggingface\hub\models--Salesforce--blip-image-captioning-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either n

In [5]:
def extraer_frames(video_path, intervalo_segundos=5, carpeta_salida="frames_extraidos"):
    if not os.path.exists(carpeta_salida):
        os.makedirs(carpeta_salida)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duracion = total_frames / fps

    print(f"Duración del video: {duracion:.2f} segundos - FPS: {fps}")

    frames_extraidos = []
    frame_num = 0
    while cap.isOpened():
        tiempo_actual = frame_num / fps
        if tiempo_actual > duracion:
            break
        cap.set(cv2.CAP_PROP_POS_MSEC, tiempo_actual * 1000)
        ret, frame = cap.read()
        if ret:
            frame_name = os.path.join(carpeta_salida, f"frame_{int(tiempo_actual)}s.jpg")
            cv2.imwrite(frame_name, frame)
            frames_extraidos.append(frame_name)
        else:
            break
        frame_num += int(intervalo_segundos * fps)

    cap.release()
    return frames_extraidos


In [ ]:
def guardar_conversacion_pdf(chat_historial, nombre_base="conversacion"):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    nombre = f"{nombre_base}_{timestamp}.pdf"
    c = canvas.Canvas(nombre, pagesize=letter)
    width, height = letter
    y = height - 40

    c.setFont("Helvetica-Bold", 14)
    c.drawString(40, y, "Conversación con Asistente Educativo")
    y -= 30

    c.setFont("Helvetica", 10)
    for i, (user, assistant) in enumerate(chat_historial):
        for prefix, texto in [("Usuario:", user), ("Asistente:", assistant)]:
            for linea in texto.split("\n"):
                if y < 40:
                    c.showPage()
                    y = height - 40
                c.drawString(40, y, f"{prefix} {linea.strip()}")
                y -= 15
            y -= 10

    c.setFont("Helvetica-Oblique", 8)
    c.drawString(40, 20, f"Generado: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    c.save()
    return nombre


In [3]:
def extraer_audio_con_ffmpeg(video_path, output_audio_path="audio_extraido.wav"):
    try:
        (
            ffmpeg
            .input(video_path)
            .output(output_audio_path, format='wav', acodec='pcm_s16le', ac=1, ar='16000')
            .overwrite_output()
            .run(quiet=True)
        )
        return output_audio_path
    except ffmpeg.Error as e:
        return None




def responder_con_gpt4(pregunta, contexto):
    prompt = f"""
Eres un asistente educativo que responde en formato Markdown. Usa listas, bloques de código y **fórmulas matemáticas con doble signo de dólar `$$`** para que se rendericen correctamente.

--- CONTEXTO TRANSCRITO ---
{contexto}

--- PREGUNTA DEL USUARIO ---
{pregunta}

Responde usando **Markdown** con formato matemático en bloque usando `$$`.
"""
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=800
    )
    return response.choices[0].message.content


In [4]:
def interfaz_gradio(archivo, pregunta_usuario):
    if archivo is None:
        return "⚠️ No se proporcionó ningún archivo.", ""
    
    ext = os.path.splitext(archivo)[1].lower()

    if ext in [".mp3", ".wav", ".m4a"]:
        archivo_audio = archivo
    elif ext in [".mp4", ".mov", ".avi", ".mkv"]:
        archivo_audio = extraer_audio_con_ffmpeg(archivo)
        if archivo_audio is None:
            return "⚠️ No se pudo extraer el audio del video.", ""
    else:
        return f"⚠️ Tipo de archivo no soportado: {ext}", ""

    transcripcion = transcribir_audio(archivo_audio)

    if not pregunta_usuario.strip():
        return transcripcion, "💡 Escribe una pregunta para recibir una respuesta."

    respuesta = responder_con_gpt4(pregunta_usuario, transcripcion)
    
    return transcripcion, respuesta


In [ ]:
with gr.Blocks(title="Asistente Multimodal con GPT-4") as demo:
    gr.Markdown("## 🎓 Asistente Multimodal con GPT-4")
    gr.Markdown("Sube un audio o video, pregunta lo que quieras y GPT-4 te responde con soporte para código, listas y fórmulas matemáticas en Markdown.")

    archivo = gr.File(label="📁 Archivo (audio o video)", file_types=[".mp3", ".wav", ".m4a", ".mp4", ".mov", ".avi", ".mkv"])
    chatbot = gr.Chatbot(label="🗨️ Conversación")
    pregunta = gr.Textbox(label="💬 Escribe tu pregunta", placeholder="Ej. ¿Qué se dijo en la explicación?", lines=1)
    estado = gr.Markdown("🟡 Esperando pregunta...")

    enviar = gr.Button("🚀 Enviar")
    limpiar = gr.Button("🧹 Limpiar conversación")
    btn_pdf = gr.Button("📄 Guardar conversación como PDF")
    archivo_pdf_gr = gr.File(label="📄 Descarga tu conversación")

    # Funciones conectadas
    def limpiar_historial():
        return [], "🟡 Conversación reiniciada."

    def exportar_pdf(chat_historial):
        archivo_pdf = guardar_conversacion_pdf(chat_historial)
        return gr.File.update(value=archivo_pdf)

    # Botones
    limpiar.click(fn=limpiar_historial, outputs=[chatbot, estado])
    btn_pdf.click(fn=exportar_pdf, inputs=[chatbot], outputs=[archivo_pdf_gr])

    # Acciones del chatbot (asegúrate de tener la función `conversar`)
    enviar.click(fn=conversar, inputs=[archivo, pregunta, chatbot], outputs=[chatbot, estado])
    pregunta.submit(fn=conversar, inputs=[archivo, pregunta, chatbot], outputs=[chatbot, estado])

    def conversar(archivo, mensaje_usuario, chat_historial):
        if archivo is None:
            return chat_historial, "⚠️ Por favor sube un archivo primero."

        ext = os.path.splitext(archivo)[1].lower()

        if ext in [".mp3", ".wav", ".m4a"]:
            archivo_audio = archivo
        elif ext in [".mp4", ".mov", ".avi", ".mkv"]:
            archivo_audio = extraer_audio_con_ffmpeg(archivo)
            if archivo_audio is None:
                return chat_historial, "❌ No se pudo extraer el audio del video."
        else:
            return chat_historial, f"❌ Formato no soportado: {ext}"

        transcripcion = obtener_o_transcribir_audio(archivo_audio)

        mensajes = [{"role": "system", "content": "Eres un asistente educativo que responde en formato Markdown."}]
        for user_msg, assistant_msg in chat_historial:
            mensajes.append({"role": "user", "content": user_msg})
            mensajes.append({"role": "assistant", "content": assistant_msg})

        if not chat_historial:
            mensaje_completo = f"{mensaje_usuario}\n\nCON CONTEXTO:\n{transcripcion}"
        else:
            mensaje_completo = mensaje_usuario

        mensajes.append({"role": "user", "content": mensaje_completo})

        respuesta = openai.chat.completions.create(
            model="gpt-4",
            messages=mensajes,
            max_tokens=700
        ).choices[0].message.content

        chat_historial.append((mensaje_usuario, respuesta))
        return chat_historial, "✅ Respondido"

    enviar.click(fn=conversar, inputs=[archivo, pregunta, chatbot], outputs=[chatbot, estado])
    pregunta.submit(fn=conversar, inputs=[archivo, pregunta, chatbot], outputs=[chatbot, estado])

demo.launch()

C:\Users\Arria\AppData\Local\Temp\ipykernel_37828\1126542823.py:6: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="🗨️ Conversación")


* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
